# Partial cross entropy for point supervised segmentation

Building segmentation on SpaceNet 1 (Rio de Janeiro) trained from a few labelled pixels per tile.

Runtime > Change runtime type > T4 GPU

In [ ]:
!pip install -q rasterio

In [ ]:
import io
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import requests
import torch
import torch.nn.functional as F
from PIL import Image
from rasterio.features import rasterize
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA = Path('data/rio')
RUNS = Path('runs')
IGNORE_INDEX = 255
NUM_CLASSES = 2
TILE_SIZE = 256
NUM_TILES = 900
EPOCHS = 40
BATCH_SIZE = 16
LR = 3e-4
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print(DEVICE, torch.__version__)

## Dataset

Pansharpened WorldView-3 tiles at 0.5 m with building footprints, taken from the public
SpaceNet bucket. The footprints are rasterised into binary masks. About 10 minutes.

In [ ]:
BUCKET = 'https://spacenet-dataset.s3.amazonaws.com/spacenet/SN1_buildings/train'
MAX_IMAGE_ID = 6940
MIN_BUILDING_FRAC = 0.01


def fetch(url, attempts=4):
    for _ in range(attempts):
        try:
            response = requests.get(url, timeout=60)
        except requests.RequestException:
            continue
        if response.status_code == 404:
            return None
        if response.status_code == 200:
            return response.content
    return None


def build_tile(image_id):
    label = fetch(f'{BUCKET}/geojson/Geo_AOI_1_RIO_img{image_id}.geojson')
    if label is None:
        return None
    features = json.loads(label)['features']
    if not features:
        return None

    raw = fetch(f'{BUCKET}/3band/3band_AOI_1_RIO_img{image_id}.tif')
    if raw is None:
        return None

    with rasterio.open(io.BytesIO(raw)) as src:
        image = src.read().transpose(1, 2, 0)
        mask = rasterize([f['geometry'] for f in features], out_shape=(src.height, src.width),
                         transform=src.transform, dtype='uint8')

    if image.shape[2] != 3 or image.max() == 0 or mask.mean() < MIN_BUILDING_FRAC:
        return None

    side = min(image.shape[0], image.shape[1])
    top = (image.shape[0] - side) // 2
    left = (image.shape[1] - side) // 2
    image = Image.fromarray(image[top:top + side, left:left + side])
    mask = Image.fromarray(mask[top:top + side, left:left + side] * 255)
    return image.resize((TILE_SIZE, TILE_SIZE), Image.BILINEAR), mask.resize((TILE_SIZE, TILE_SIZE), Image.NEAREST)


def download_dataset(num_tiles=NUM_TILES, workers=32):
    (DATA / 'images').mkdir(parents=True, exist_ok=True)
    (DATA / 'masks').mkdir(parents=True, exist_ok=True)

    ids = list(range(1, MAX_IMAGE_ID + 1))
    random.Random(0).shuffle(ids)

    kept = []
    with ThreadPoolExecutor(workers) as pool:
        for start in range(0, len(ids), workers * 8):
            if len(kept) >= num_tiles:
                break
            chunk = ids[start:start + workers * 8]
            for image_id, tile in zip(chunk, pool.map(build_tile, chunk)):
                if tile is None or len(kept) >= num_tiles:
                    continue
                name = f'rio_{image_id:05d}.png'
                tile[0].save(DATA / 'images' / name)
                tile[1].save(DATA / 'masks' / name)
                kept.append(name)
            print(f'scanned {start + len(chunk)} ids, kept {len(kept)}', flush=True)

    kept.sort()
    (DATA / 'tiles.txt').write_text('\n'.join(kept) + '\n')


if not (DATA / 'tiles.txt').exists():
    download_dataset()

TILES = (DATA / 'tiles.txt').read_text().split()
IMAGES = {name: np.array(Image.open(DATA / 'images' / name).convert('RGB')) for name in TILES}
MASKS = {name: (np.array(Image.open(DATA / 'masks' / name)) > 127).astype(np.uint8) for name in TILES}

print(len(TILES), 'tiles, mean building cover', round(float(np.mean([m.mean() for m in MASKS.values()])), 4))

In [ ]:
shuffled = list(TILES)
np.random.default_rng(0).shuffle(shuffled)
n_val = n_test = int(len(shuffled) * 0.15)

VAL_NAMES = shuffled[:n_val]
TEST_NAMES = shuffled[n_val:n_val + n_test]
TRAIN_NAMES = shuffled[n_val + n_test:]

print(len(TRAIN_NAMES), 'train', len(VAL_NAMES), 'val', len(TEST_NAMES), 'test')

## Point annotation

In [ ]:
def sample_points(mask, points_per_image, mode, rng):
    labels = np.full(mask.shape, IGNORE_INDEX, dtype=np.uint8)
    classes = np.unique(mask)

    if mode == 'uniform':
        budget = min(points_per_image * len(classes), mask.size)
        chosen = rng.choice(mask.size, budget, replace=False)
        labels.flat[chosen] = mask.flat[chosen]
        return labels

    for cls in classes:
        candidates = np.flatnonzero(mask == cls)
        chosen = rng.choice(candidates, min(points_per_image, candidates.size), replace=False)
        labels.flat[chosen] = cls
    return labels


class RioBuildings(Dataset):
    def __init__(self, names, points_per_image=None, point_mode='balanced', augment=False, seed=0):
        self.names = names
        self.augment = augment
        self.targets = []
        for index, name in enumerate(names):
            if points_per_image is None:
                self.targets.append(MASKS[name].copy())
            else:
                rng = np.random.default_rng([seed, index])
                self.targets.append(sample_points(MASKS[name], points_per_image, point_mode, rng))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image, mask, target = IMAGES[name], MASKS[name], self.targets[index]

        if self.augment:
            k = np.random.randint(4)
            image, mask, target = (np.rot90(a, k) for a in (image, mask, target))
            if np.random.rand() < 0.5:
                image, mask, target = (np.fliplr(a) for a in (image, mask, target))

        image = (image.astype(np.float32) / 255.0 - MEAN) / STD
        return (torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(target)).long(),
                torch.from_numpy(np.ascontiguousarray(mask)).long())

In [ ]:
example = TRAIN_NAMES[0]
points = sample_points(MASKS[example], 20, 'balanced', np.random.default_rng(0))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(IMAGES[example])
axes[0].set_title('image')
axes[1].imshow(MASKS[example], cmap='gray')
axes[1].set_title('full mask')
axes[2].imshow(IMAGES[example])
for cls, colour, label in [(0, 'tab:blue', 'background'), (1, 'tab:red', 'building')]:
    ys, xs = np.where(points == cls)
    axes[2].scatter(xs, ys, s=14, c=colour, label=label)
axes[2].set_title('20 points per class')
axes[2].legend(fontsize=8, loc='lower right')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(int((points != IGNORE_INDEX).sum()), 'labelled pixels of', points.size)

## Loss

$$pfCE = \frac{\sum focal(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

In [ ]:
class PartialFocalCE(nn.Module):
    def __init__(self, gamma=0.0, class_weights=None, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        if class_weights is not None and not torch.is_tensor(class_weights):
            class_weights = torch.tensor(class_weights, dtype=torch.float32)
        self.register_buffer('class_weights', class_weights)

    def forward(self, logits, target):
        labeled = (target != self.ignore_index).float()
        target = target.masked_fill(target == self.ignore_index, 0)

        log_prob = F.log_softmax(logits, dim=1)
        log_pt = log_prob.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -((1.0 - log_pt.exp()) ** self.gamma) * log_pt

        if self.class_weights is not None:
            loss = loss * self.class_weights[target]

        return (loss * labeled).sum() / labeled.sum().clamp(min=1.0)

In [ ]:
check_logits = torch.randn(2, 3, 8, 8)
check_target = torch.randint(0, 3, (2, 8, 8))
sparse = check_target.clone()
sparse[0, 3:] = IGNORE_INDEX
sparse[1, :5] = IGNORE_INDEX
log_pt = F.log_softmax(check_logits, 1).gather(1, check_target.unsqueeze(1)).squeeze(1)

assert torch.allclose(PartialFocalCE()(check_logits, check_target),
                      F.cross_entropy(check_logits, check_target))
assert torch.allclose(PartialFocalCE()(check_logits, sparse),
                      F.cross_entropy(check_logits, sparse, ignore_index=IGNORE_INDEX))
assert torch.allclose(PartialFocalCE(gamma=2.0)(check_logits, check_target),
                      (-((1 - log_pt.exp()) ** 2) * log_pt).mean())
assert PartialFocalCE()(check_logits[:1], torch.full((1, 8, 8), IGNORE_INDEX)).item() == 0.0

print('loss checks passed')

## Model

In [ ]:
IMAGENET_URL = 'https://s3.amazonaws.com/pytorch/models/resnet18-5c106cde.pth'


def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


def upsample_to(x, reference):
    return F.interpolate(x, size=reference.shape[-2:], mode='bilinear', align_corners=False)


class UNetResNet18(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        encoder = resnet18()
        if pretrained:
            encoder.load_state_dict(torch.hub.load_state_dict_from_url(IMAGENET_URL, progress=False))

        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layer1 = encoder.layer1
        self.layer2 = encoder.layer2
        self.layer3 = encoder.layer3
        self.layer4 = encoder.layer4

        self.up4 = conv_block(512 + 256, 256)
        self.up3 = conv_block(256 + 128, 128)
        self.up2 = conv_block(128 + 64, 64)
        self.up1 = conv_block(64 + 64, 64)
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        s4 = self.layer4(s3)

        d4 = self.up4(torch.cat([upsample_to(s4, s3), s3], 1))
        d3 = self.up3(torch.cat([upsample_to(d4, s2), s2], 1))
        d2 = self.up2(torch.cat([upsample_to(d3, s1), s1], 1))
        d1 = self.up1(torch.cat([upsample_to(d2, s0), s0], 1))
        return upsample_to(self.head(d1), x)

## Training

In [ ]:
def confusion_matrix(pred, target):
    k = target * NUM_CLASSES + pred
    return np.bincount(k.ravel(), minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_confusion(cm):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / np.maximum(tp + fp + fn, 1e-9)
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9)
    return {'miou': float(iou.mean()), 'iou_background': float(iou[0]),
            'iou_building': float(iou[1]), 'f1_building': float(f1[1]),
            'overall_accuracy': float(tp.sum() / cm.sum())}


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for images, _, masks in loader:
        pred = model(images.to(DEVICE)).argmax(1).cpu().numpy()
        cm += confusion_matrix(pred, masks.numpy())
    return metrics_from_confusion(cm)


def train_run(name, points, gamma=0.0, seed=0, point_mode='balanced', epochs=EPOCHS, keep_checkpoint=False):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_set = RioBuildings(TRAIN_NAMES, points, point_mode, augment=True, seed=seed)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
    val_loader = DataLoader(RioBuildings(VAL_NAMES), batch_size=BATCH_SIZE, num_workers=2)
    test_loader = DataLoader(RioBuildings(TEST_NAMES), batch_size=BATCH_SIZE, num_workers=2)

    model = UNetResNet18().to(DEVICE)
    criterion = PartialFocalCE(gamma=gamma).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    labelled = sum(int((t != IGNORE_INDEX).sum()) for t in train_set.targets)
    total = sum(t.size for t in train_set.targets)
    print(f'{name}: {labelled}/{total} labelled pixels ({100 * labelled / total:.4f}%)', flush=True)

    history, best, best_state = [], {'miou': -1.0}, None
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for images, targets, _ in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(images.to(DEVICE)), targets.to(DEVICE))
            loss.backward()
            optimizer.step()
            running += loss.item()
        scheduler.step()

        val = evaluate(model, val_loader)
        history.append({'epoch': epoch, 'loss': running / len(train_loader), **val})
        if val['miou'] > best['miou']:
            best = val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch:02d}  loss {history[-1]['loss']:.4f}  "
                  f"val mIoU {val['miou']:.4f}  building IoU {val['iou_building']:.4f}", flush=True)

    model.load_state_dict(best_state)
    test = evaluate(model, test_loader)
    print(f"  test mIoU {test['miou']:.4f}  building IoU {test['iou_building']:.4f}  "
          f"[{(time.time() - start) / 60:.1f} min]", flush=True)

    result = {'name': name, 'points': points, 'gamma': gamma, 'seed': seed,
              'labelled_fraction': labelled / total, 'history': history,
              'val': best, 'test': test, 'minutes': (time.time() - start) / 60}
    RUNS.mkdir(exist_ok=True)
    (RUNS / f'{name}.json').write_text(json.dumps(result, indent=2))
    if keep_checkpoint:
        torch.save(best_state, RUNS / f'{name}.pt')
    return result

In [ ]:
demo = train_run('demo_points5', points=5, epochs=5)

## Experiments

Factor 1 is the annotation budget, factor 2 is the focal gamma. The last two runs repeat one
setting with different seeds to measure how much of any difference is noise.

In [ ]:
GRID = [
    ('dense_ce', 0, 0.0, 0),
    ('points1_ce', 1, 0.0, 0),
    ('points5_ce', 5, 0.0, 0),
    ('points20_ce', 20, 0.0, 0),
    ('points100_ce', 100, 0.0, 0),
    ('points1_focal2', 1, 2.0, 0),
    ('points5_focal2', 5, 2.0, 0),
    ('points20_focal2', 20, 2.0, 0),
    ('points5_ce_seed1', 5, 0.0, 1),
    ('points5_ce_seed2', 5, 0.0, 2),
]
CHECKPOINTS = {'dense_ce', 'points5_ce'}

results = {}
for name, points, gamma, seed in GRID:
    saved = RUNS / f'{name}.json'
    if saved.exists():
        results[name] = json.loads(saved.read_text())
        continue
    results[name] = train_run(name, points if points else None, gamma, seed,
                              keep_checkpoint=name in CHECKPOINTS)

## Results

In [ ]:
import pandas as pd

table = pd.DataFrame([{
    'run': r['name'],
    'points': r['points'],
    'gamma': r['gamma'],
    'seed': r['seed'],
    'labelled %': round(100 * r['labelled_fraction'], 4),
    'test mIoU': round(r['test']['miou'], 4),
    'building IoU': round(r['test']['iou_building'], 4),
    'building F1': round(r['test']['f1_building'], 4),
    'accuracy': round(r['test']['overall_accuracy'], 4),
} for r in results.values()]).sort_values(['gamma', 'points', 'seed'])

table.to_csv('results.csv', index=False)
table

In [ ]:
density = [1, 5, 20, 100]
ce = [results[f'points{n}_ce']['test']['iou_building'] for n in density]
focal = [results[f'points{n}_focal2']['test']['iou_building'] for n in density[:3]]
dense = results['dense_ce']['test']['iou_building']
seeds = [results[n]['test']['iou_building'] for n in ['points5_ce', 'points5_ce_seed1', 'points5_ce_seed2']]

fig, ax = plt.subplots(figsize=(6, 4))
ax.axhline(dense, color='grey', ls='--', label=f'full masks ({dense:.3f})')
ax.vlines(5, min(seeds), max(seeds), color='tab:blue', alpha=0.35, lw=6,
          label=f'seed spread at 5 points ({max(seeds) - min(seeds):.3f})')
ax.plot(density, ce, 'o-', label='partial CE')
ax.plot(density[:3], focal, 's-', label='partial focal CE')
ax.set_xscale('log')
ax.set_xlabel('labelled points per class per tile')
ax.set_ylabel('building IoU (test)')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('seed mean %.4f, spread %.4f' % (np.mean(seeds), max(seeds) - min(seeds)))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name in ['dense_ce', 'points1_ce', 'points5_ce', 'points20_ce', 'points100_ce']:
    history = results[name]['history']
    ax.plot([h['epoch'] for h in history], [h['miou'] for h in history], label=name)
ax.set_xlabel('epoch')
ax.set_ylabel('val mIoU')
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
subset = TEST_NAMES[:4]
loader = DataLoader(RioBuildings(subset), batch_size=len(subset))
images, _, masks = next(iter(loader))

predictions = {}
for name in sorted(CHECKPOINTS):
    model = UNetResNet18(pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / f'{name}.pt', map_location=DEVICE, weights_only=True))
    model.eval()
    with torch.no_grad():
        predictions[name] = model(images.to(DEVICE)).argmax(1).cpu().numpy()

columns = ['image', 'ground truth'] + list(predictions)
fig, axes = plt.subplots(len(subset), len(columns), figsize=(3 * len(columns), 3 * len(subset)))
for row, name in enumerate(subset):
    axes[row, 0].imshow(IMAGES[name])
    axes[row, 1].imshow(masks[row], cmap='gray')
    for col, key in enumerate(predictions):
        axes[row, 2 + col].imshow(predictions[key][row], cmap='gray')
for ax, title in zip(axes[0], columns):
    ax.set_title(title)
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()